# Sparkle V2 — Fase 3.6d: evaluación del backbone mejorado

**Protocolo:** `Protocolo_Fase36_Camino_al_70.md` **Enmienda 3**.
**Requiere:** `dataset_fase3_6_raw_v2.npz` (de 3.6c) y `oof_predicciones.npz` (de 3.5b).

Reproduce el pipeline completo de §11-§13 cambiando **solo** la rama de apariencia:
OOF por folds → proyección d=16 → resumen media+desvío → cabeza logística → ventanas
k=5 span≤7 → curva riesgo-cobertura. Mismos folds, misma semilla, misma limpieza, mismo
umbral, misma grilla.

**Criterio (§0 + Enmienda 1):** accuracy ≥ 0,70 con cobertura ≥ 0,60 **y** límite inferior del
IC de balanced accuracy ≥ 0,5896. Además, comparación **pareada** contra la configuración
actual, que es la que dice si el backbone aportó.

**El Test no se toca.**

In [ ]:
import sys, os, json, collections, time
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
else:
    PROJECT_DIR = os.environ.get('SPARKLE_PROJECT_DIR', './SparkleV2')

CACHE_32 = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_2_alineado.npz'
CACHE_V2 = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_6_raw_v2.npz'
PROJ_V2  = f'{PROJECT_DIR}/checkpoints_fase3_6/proyeccion_v2.npz'
OOF_OLD  = f'{PROJECT_DIR}/fase3_5_proyeccion/oof_predicciones.npz'
OUT_36   = f'{PROJECT_DIR}/fase3_6_camino70'
os.makedirs(OUT_36, exist_ok=True)

import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# ------- CONGELADOS -------
SEED, UMBRAL, N_FOLDS = 42, 0.50, 5
D_PROJ, K, SPAN_MAX   = 16, 5, 7
COBERTURAS   = np.round(np.arange(1.00, 0.499, -0.05), 2)
ACC_OBJETIVO, COB_MIN, BALACC_PISO = 0.70, 0.60, 0.5896
N_BOOT, N_FRAMES = 5000, 60
DELTA_FEATURES = ['ear_avg', 'yaw', 'pitch', 'mar', 'jawOpen', 'eyeBlinkLeft', 'eyeBlinkRight',
                  'eyeLookInLeft', 'eyeLookOutLeft', 'browInnerUp']

for p in (CACHE_32, CACHE_V2, PROJ_V2, OOF_OLD):
    assert os.path.exists(p), f'Falta {p}'
d32, dv2 = np.load(CACHE_32, allow_pickle=True), np.load(CACHE_V2, allow_pickle=True)
pz = np.load(PROJ_V2); W_v2, b_v2, W_alt = pz['W'], pz['b'], pz['W_alt']
zold = np.load(OOF_OLD, allow_pickle=True)
DIM_RAW = int(dv2['dim_raw'])
FEATS22 = [str(f) for f in d32['feature_names']]
ids_of = lambda a: np.array([str(c) for c in a])
for k in ('tr', 'va', 'te'):
    assert (ids_of(d32[f'id_{k}']) == ids_of(dv2[f'id_{k}'])).all(), f'ClipIDs desalineados en {k}'
print(f'✅ Alineación verificada · backbone v2: α={float(dv2["alpha"])} @{int(dv2["crop"])} px '
      f'· {DIM_RAW}-d · proyección {W_v2.shape}')

Xg_pool = np.concatenate([d32['Xg_tr'], d32['Xg_va']])
Xv_pool = np.concatenate([dv2['Xr_tr'], dv2['Xr_va']])
y3_pool = np.concatenate([d32['y3_tr'], d32['y3_va']])
id_pool = np.concatenate([ids_of(d32['id_tr']), ids_of(d32['id_va'])])
y_pool  = (y3_pool > 0).astype(np.int64)
part_pool = np.array([c[:6] for c in id_pool])
print(f'Pool: {len(y_pool)} clips · {len(set(part_pool))} participantes')

## 1. Preprocesamiento y utilidades — idénticos a 3.5b/3.6b

In [ ]:
def interpolate_sequence(seq):
    df = pd.DataFrame(seq).interpolate(method='linear', axis=0, limit_direction='both')
    return df.ffill().bfill().values.astype(np.float32)

def clean(X, medians=None, thresh=0.30):
    keep, out = [], []
    for i in range(len(X)):
        if np.isnan(X[i]).any(axis=1).mean() > thresh:
            continue
        s = interpolate_sequence(X[i])
        if np.isnan(s).any() and medians is not None:
            idx = np.where(np.isnan(s)); s[idx] = np.take(medians, idx[1])
        out.append(s); keep.append(i)
    return np.stack(out), np.array(keep)

delta_idx = [FEATS22.index(f) for f in DELTA_FEATURES]
def add_deltas(X):
    d_ = np.zeros_like(X[:, :, delta_idx]); d_[:, 1:, :] = np.diff(X[:, :, delta_idx], axis=1)
    return np.concatenate([X, d_], axis=-1)

def resumir(X):
    X = np.nan_to_num(X); return np.hstack([X.mean(axis=1), X.std(axis=1)])

def metricas(y, p, umbral=UMBRAL):
    pred = (p > umbral).astype(int); triv = max((y == 0).mean(), (y == 1).mean())
    acc = accuracy_score(y, pred)
    return {'n': int(len(y)), 'trivial': round(float(triv), 4),
            'accuracy': round(float(acc), 4), 'brecha': round(float(acc - triv), 4),
            'balanced_acc': round(float(balanced_accuracy_score(y, pred)), 4),
            'f1_macro': round(float(f1_score(y, pred, average='macro')), 4),
            'auc': round(float(roc_auc_score(y, p)), 4)}

def _ba(y, ok):
    n0, n1 = (y == 0).sum(), (y == 1).sum()
    return np.nan if (n0 == 0 or n1 == 0) else 0.5*(ok[y==0].sum()/n0 + ok[y==1].sum()/n1)

## 2. Bucle OOF con la apariencia nueva

`W_proj` parametriza la proyección para poder reusar el bucle en el control de robustez sobre
5 matrices alternativas (Enmienda 3).

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(sgkf.split(Xg_pool, y_pool, groups=part_pool))

def correr_oof(W_proj, b_proj):
    oof = np.full(len(y_pool), np.nan); mask = np.zeros(len(y_pool), bool)
    for fold, (tr, va) in enumerate(folds):
        assert not (set(part_pool[tr]) & set(part_pool[va])), f'LEAKAGE fold {fold}'  # regla 7
        g_med = np.nanmedian(Xg_pool[tr].reshape(-1, 22), axis=0)
        Xg_a, kga = clean(Xg_pool[tr], g_med); Xg_b, kgb = clean(Xg_pool[va], g_med)
        Ea, kea = clean(Xv_pool[tr].astype(np.float32), None, thresh=0.5)
        Eb, keb = clean(Xv_pool[va].astype(np.float32), None, thresh=0.5)
        ka = np.intersect1d(kga, kea); kb = np.intersect1d(kgb, keb)
        ig = {k: i for i, k in enumerate(kga)}; ie = {k: i for i, k in enumerate(kea)}
        jg = {k: i for i, k in enumerate(kgb)}; je = {k: i for i, k in enumerate(keb)}
        gsc = StandardScaler().fit(add_deltas(Xg_a).reshape(-1, 32))
        sg = lambda X: gsc.transform(X.reshape(-1, 32)).reshape(len(X), N_FRAMES, 32)
        P = lambda E: (E @ W_proj + b_proj)
        R_tr = np.hstack([resumir(sg(add_deltas(Xg_a))[[ig[k] for k in ka]]),
                          resumir(P(Ea[[ie[k] for k in ka]]))])
        R_ev = np.hstack([resumir(sg(add_deltas(Xg_b))[[jg[k] for k in kb]]),
                          resumir(P(Eb[[je[k] for k in kb]]))])
        sc = StandardScaler().fit(R_tr)
        lr = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED)
        lr.fit(sc.transform(R_tr), y_pool[tr][ka])
        idx = va[kb]; oof[idx] = lr.predict_proba(sc.transform(R_ev))[:, 1]; mask[idx] = True
    assert not np.isnan(oof[mask]).any() and np.isnan(oof[~mask]).all()
    return oof, mask

t0 = time.time()
oof_v2, mask = correr_oof(W_v2, b_v2)
print(f'OOF v2 listo en {(time.time()-t0)/60:.1f} min · {mask.sum()} clips')

# Alinear con el OOF viejo (mismo subconjunto de clips)
ids_old = np.array([str(c) for c in zold['ids']])
ids_new = id_pool[mask]
assert set(ids_new) == set(ids_old), (
    f'El subconjunto OOF cambió: {len(set(ids_new) ^ set(ids_old))} clips difieren. '
    'La detección facial debería ser idéntica; revisar antes de comparar.')
orden = {c: i for i, c in enumerate(ids_new)}
sel = np.array([orden[c] for c in ids_old])          # reordenar v2 al orden del viejo
p_v2 = oof_v2[mask][sel]
y_c   = zold['y']; part_c = np.array([str(p) for p in zold['part']]); ids_c = ids_old
p_old = zold['R1_d16']
print(f'✅ Mismo subconjunto de {len(ids_c)} clips en ambas configuraciones')

## 3. Nivel clip — comparación pareada

In [ ]:
print('ACTUAL (α=0,35 @112):', metricas(y_c, p_old))
print('NUEVO  (α=1,0  @128):', metricas(y_c, p_v2))

def boot_pareado(pa, pb, y, part, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); parts = np.unique(part)
    idx = {q: np.where(part == q)[0] for q in parts}
    oka = (pa > UMBRAL).astype(int) == y; okb = (pb > UMBRAL).astype(int) == y
    d = np.empty(n_boot)
    for i in range(n_boot):
        j = np.concatenate([idx[q] for q in rng.choice(parts, size=len(parts), replace=True)])
        d[i] = _ba(y[j], okb[j]) - _ba(y[j], oka[j])
    d = d[~np.isnan(d)]; lo, hi = np.percentile(d, [2.5, 97.5])
    return {'delta_balacc': round(float(d.mean()), 4),
            'IC95': [round(float(lo), 4), round(float(hi), 4)], 'cruza_cero': bool(lo <= 0 <= hi)}

r_clip = boot_pareado(p_old, p_v2, y_c, part_c)
print('\nBalAcc nuevo − actual (clip):', r_clip)
if r_clip['cruza_cero']:
    print('⚠️  El IC cruza cero: el backbone mejorado NO aporta de forma demostrable (regla 6).')

# Control de robustez: 5 proyecciones alternativas (no es el resultado principal)
bas = []
for s in range(len(W_alt)):
    o, m = correr_oof(W_alt[s], np.zeros(D_PROJ, np.float32))
    bas.append(balanced_accuracy_score(y_c, (o[m][sel] > UMBRAL).astype(int)))
print(f'Robustez sobre 5 proyecciones: BalAcc {np.mean(bas):.4f} ± {np.std(bas):.4f} '
      f'(principal, semilla 42: {balanced_accuracy_score(y_c,(p_v2>UMBRAL).astype(int)):.4f})')

## 4. Ventanas k=5 span≤7 y curva riesgo-cobertura

In [ ]:
def parse(cid):
    c = os.path.splitext(str(cid).strip())[0]; return c[:6], int(c[6:])

def ventanas_con_huecos(ids, k=K, span_max=SPAN_MAX):
    por = collections.defaultdict(list)
    for i, cid in enumerate(ids):
        s, t = parse(cid); por[s].append((t, i))
    wins = []
    for s, pares in por.items():
        pares.sort(); i = 0
        while i + k - 1 < len(pares):
            v = pares[i:i+k]
            if v[-1][0] - v[0][0] <= span_max - 1:
                wins.append([j for _, j in v]); i += k
            else:
                i += 1
    return wins

wins = ventanas_con_huecos(ids_c)
for w in wins:
    assert len({parse(ids_c[i])[0] for i in w}) == 1
    ts = sorted(parse(ids_c[i])[1] for i in w); assert ts[-1]-ts[0] <= SPAN_MAX-1
print(f'{len(wins)} ventanas · cobertura {sum(len(w) for w in wins)/len(ids_c):.1%}')

def agregar(wins, y, p, ids):
    return (np.array([int(y[w].sum()*2 > len(w)) for w in wins]),
            np.array([float(p[w].mean()) for w in wins]),
            np.array([parse(ids[w[0]])[0] for w in wins]))

y_w, pw_v2, part_w = agregar(wins, y_c, p_v2, ids_c)
_,   pw_old, _     = agregar(wins, y_c, p_old, ids_c)

def curva(p, y):
    conf = np.abs(p - UMBRAL); orden = np.argsort(-conf); filas = []
    for c in COBERTURAS:
        k = max(1, int(round(c*len(p)))); s = orden[:k]
        filas.append({'cobertura': float(c), **metricas(y[s], p[s])})
    return pd.DataFrame(filas)

df_v2, df_old = curva(pw_v2, y_w), curva(pw_old, y_w)
print('\n=== VENTANA · backbone NUEVO ==='); print(df_v2.to_string(index=False))
print('\n=== VENTANA · backbone ACTUAL (referencia §13) ==='); print(df_old.to_string(index=False))
df_v2.to_csv(f'{OUT_36}/curva_ventana_backbone_v2.csv', index=False)

## 5. IC y comparación pareada a nivel ventana

In [ ]:
def boot_cob(p, y, part, c, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); parts = np.unique(part)
    idx = {q: np.where(part == q)[0] for q in parts}; conf = np.abs(p - UMBRAL)
    acc, ba = np.empty(n_boot), np.empty(n_boot)
    for i in range(n_boot):
        j = np.concatenate([idx[q] for q in rng.choice(parts, size=len(parts), replace=True)])
        j = j[np.argsort(-conf[j])[:max(1, int(round(c*len(j))))]]
        ys = y[j]; ok = ((p[j] > UMBRAL).astype(int) == ys)
        acc[i] = ok.mean(); ba[i] = _ba(ys, ok)
    pct = lambda v: [round(float(x), 4) for x in np.percentile(v[~np.isnan(v)], [2.5, 97.5])]
    return {'accuracy_IC95': pct(acc), 'balacc_IC95': pct(ba)}

for c in (1.00, 0.70, 0.65, 0.60):
    f = df_v2[df_v2.cobertura == c].iloc[0]; r = boot_cob(pw_v2, y_w, part_w, c)
    print(f'c={c:.2f} | acc {f.accuracy:.4f} {r["accuracy_IC95"]} | '
          f'BalAcc {f.balanced_acc:.4f} {r["balacc_IC95"]} | trivial {f.trivial:.4f}')

r_win = boot_pareado(pw_old, pw_v2, y_w, part_w)
print('\nBalAcc nuevo − actual (ventana, cobertura 1,00):', r_win)

## 6. Veredicto

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for d_, lab in ((df_v2, 'backbone nuevo'), (df_old, 'actual')):
    ax[0].plot(d_.cobertura, d_.accuracy, 'o-', label=f'accuracy {lab}')
    ax[1].plot(d_.cobertura, d_.balanced_acc, 'o-', label=f'BalAcc {lab}')
ax[0].plot(df_v2.cobertura, df_v2.trivial, 's--', alpha=.5, color='gray', label='trivial')
ax[0].axhline(ACC_OBJETIVO, color='r', ls=':'); ax[1].axhline(BALACC_PISO, color='r', ls=':')
for a in ax:
    a.axvline(COB_MIN, color='gray', ls=':'); a.invert_xaxis()
    a.legend(fontsize=8); a.set_xlabel('cobertura')
plt.tight_layout(); plt.savefig(f'{OUT_36}/backbone_v2_vs_actual.png', dpi=120); plt.show()

print('================ VEREDICTO PASO 3 ================')
cands = []
for _, f in df_v2[df_v2.cobertura >= COB_MIN].iterrows():
    if f.accuracy >= ACC_OBJETIVO:
        r = boot_cob(pw_v2, y_w, part_w, f.cobertura)
        if r['balacc_IC95'][0] >= BALACC_PISO:
            cands.append({'cobertura': f.cobertura, 'accuracy': f.accuracy,
                          'trivial': f.trivial, 'brecha': f.brecha,
                          'balanced_acc': f.balanced_acc, **r})
if not cands:
    mejor = df_v2[df_v2.cobertura >= COB_MIN].sort_values('accuracy').iloc[-1]
    print(f'❌ No se alcanza accuracy ≥ {ACC_OBJETIVO} con cobertura ≥ {COB_MIN}.')
    print(f'   Mejor admisible: {mejor.accuracy:.4f} a cobertura {mejor.cobertura:.2f} '
          f'(BalAcc {mejor.balanced_acc:.4f}, trivial {mejor.trivial:.4f}).')
    print(f'   Referencia §13 (backbone actual): 0,6919 a cobertura 0,65.')
    print('   Las tres vías de Fase 3.6 quedan agotadas. Decidir el cierre con la evidencia.')
else:
    e = sorted(cands, key=lambda r: -r['cobertura'])[0]
    print('✅ Cumple. Se elige la MAYOR cobertura que satisface, no la mejor:')
    print(json.dumps(e, indent=2, ensure_ascii=False))
    print('\n⚠️  Cinco decisiones tomadas sobre OOF (d, cobertura, k, span, backbone). El número '
          'de Test vendrá por debajo de éste y su IC medirá ~±7 pts con 19 participantes '
          '(§13.5). Congelar TODO y escribir la enmienda de cierre antes de abrirlo.')

json.dump({'clip': {'actual': metricas(y_c, p_old), 'nuevo': metricas(y_c, p_v2),
                    'pareado': r_clip},
           'ventana': {'curva_nuevo': df_v2.to_dict('records'), 'pareado': r_win},
           'robustez_proyeccion': {'media': float(np.mean(bas)), 'sigma': float(np.std(bas))},
           'candidatas': cands},
          open(f'{OUT_36}/paso3_backbone.json', 'w'), indent=2, ensure_ascii=False, default=str)
print('\nEvaluaciones sobre Test consumidas: 0.')